# Chapter 09: Multimodal Models

Multimodal models process **more than one type of input** — text, images, audio, or video.

We will explore:
1. Image classification (vision model)
2. Image captioning (vision → text)
3. Visual question answering (VQA)
4. Optical character recognition (OCR)

## Setup

In [ ]:
# !pip install transformers torch pillow requests
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

def load_image(url):
    r = requests.get(url, timeout=10)
    return Image.open(BytesIO(r.content)).convert('RGB')

print('Ready')

## Part 1 — Image Classification

In [ ]:
clf = pipeline("image-classification", model="google/vit-base-patch16-224")

img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg"
img = load_image(img_url)
img.save("/tmp/test_dog.jpg")
print("Image saved to /tmp/test_dog.jpg")

results = clf(img)
for r in results[:5]:
    print(f"  {r['label']:30s}  {r['score']:.3f}")

## Part 2 — Image Captioning

In [ ]:
captioner = pipeline("image-to-text", model="nlpconnect/vit-gpt2-image-captioning")

caption = captioner(img)[0]["generated_text"]
print("Caption:", caption)

## Part 3 — Visual Question Answering (VQA)

In [ ]:
vqa = pipeline("visual-question-answering", model="dandelin/vilt-b32-finetuned-vqa")

questions = ["What animal is in the image?", "What color is it?", "Is it indoors or outdoors?"]
for q in questions:
    ans = vqa(img, q)
    print(f"Q: {q}")
    print(f"A: {ans[0]['answer']} (score={ans[0]['score']:.3f})")
    print()

## Part 4 — Zero-shot image classification

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

candidate_labels = ["a dog", "a cat", "a car", "a building", "a person"]
inputs = processor(text=candidate_labels, images=img, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image[0]
    probs = logits.softmax(dim=0)

for label, prob in sorted(zip(candidate_labels, probs), key=lambda x: -x[1]):
    print(f"  {label:20s}  {prob:.3f}")

## Summary

| Task | Model |
|------|-------|
| Classification | ViT (Vision Transformer) |
| Captioning | ViT + GPT-2 |
| VQA | ViLT |
| Zero-shot | CLIP |

All these models follow the same pattern: an image encoder produces a vector, then a text decoder or classifier produces the output.